# nnlab Getting Started

This notebook introduces the basic building blocks of nnlab.

We will construct a small neural network pipeline manually:

1. Define a transition kernel
2. Build a parameterized activation function
3. Create a dense layer
4. Apply the activation through an activation layer
5. Generate predictions
6. Measure error using a loss function
7. Visualize the behavior

The goal is not training yet, but understanding how the
components interact.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nnlab.kernels import LogisticKernel

from nnlab.activations import ParameterizedActivation

from nnlab.layers import ( Dense, ActivationLayer )

from nnlab.models import FeedForward

from nnlab.losses import MeanSquaredError 

from nnlab.visualization import plot_activation 

## Step 1: Transition Kernel

The transition kernel defines the mathematical shape of the
activation transformation.

Here we use a logistic transition:

$$
K(x)=\frac{1}{1+e^{-x}}
$$

The kernel itself does not know about neural networks.
It only defines a mathematical response curve.

In [ ]:
kernel = LogisticKernel()

x = np.linspace( -7.5, 7.5, 500 )
y = kernel.forward(x)

plt.figure(figsize=(8,4))

plt.plot( x,  y )
plt.title( "Logistic Transition Kernel" )
plt.xlabel( "Input" )
plt.ylabel( "Kernel Response" )
plt.grid(True)

plt.show()

## Step 2: Parameterized Activation

The activation wraps the kernel and introduces parameters:

$$
\phi(x)=aK(\frac{x-c}{w})+b
$$

where:

- $c$ controls center
- $w$ controls width
- $a$ controls scale
- $b$ controls offset

In [ ]:
activation = ParameterizedActivation(
    kernel=kernel,
    center=0.0,
    kernel_scale=0.50,
    amplitude=1.0,
    bias=0.0,
)

plot_activation( activation )

## Step 3: Dense Layer

A dense layer performs:

$$
y = Wx+b
$$

The layer performs a linear projection.
The activation function will introduce non-linearity.

In [ ]:
dense = Dense(
    input_size=1,
    output_size=1,
)

dense.weights = np.array( [ [2.0] ] )
dense.bias = np.array( [ 0.5 ] )

activation_layer = ActivationLayer(
    activation=activation,
)

## Step 4: Compose Layers

A neural network is a composition of transformations.

Our network:

$$
x
\rightarrow Wx+b
\rightarrow \phi(x)
$$

In [ ]:
model = FeedForward( layers=[ dense, activation_layer, ] )

x = np.linspace( -3, 3, 200 ).reshape( -1, 1 )

prediction = model.forward( x )
prediction.shape

In [ ]:
plt.figure(figsize=(8,4))

plt.plot( x, prediction )

plt.title( "Network Forward Projection" )
plt.xlabel( "Input" )
plt.ylabel( "Output" )
plt.grid(True)

plt.show()

## Step 5: Evaluate Error

A loss function measures the difference between
the network output and the desired target.

Here we create a simple target function:

$$
y=x^2
$$

In [ ]:
# target = x**2
target = 1 / (1 + np.exp(-x))

plt.figure(figsize=(8,4))

plt.plot(
    x,
    target,
    label="Target",
)

plt.plot(
    x,
    prediction,
    label="Prediction",
)

plt.legend()

plt.grid(True)

plt.show()



In [ ]:
loss = MeanSquaredError()

error = loss.forward(
    prediction,
    target,
)

error

## Backward Pass

The forward pass computes a prediction:

\[
x \rightarrow model(x)
\]

The loss function measures the difference between the prediction
and the target.

The backward pass reverses this process. The loss derivative provides
the initial error signal, which is propagated backward through the
network using the chain rule.

\[
\frac{\partial L}{\partial y}
\rightarrow
activation
\rightarrow
dense
\]

At this stage, the network does not update its parameters. It only
calculates how each parameter contributed to the observed error.

In [ ]:
gradient = loss.derivative(
    prediction,
    target,
)

gradient.shape

In [ ]:
model.backward(
    gradient,
)

dense.weight_gradient
dense.bias_gradient

print(
    "Weight gradient:"
)

print(
    dense.weight_gradient,
)

print(
    "\nBias gradient:"
)

print(
    dense.bias_gradient,
)

In [ ]:
plt.figure(figsize=(8,4))

plt.plot(
    x,
    gradient,
)

plt.title(
    "Loss Gradient With Respect to Prediction"
)

plt.xlabel(
    "Input"
)

plt.ylabel(
    "dL / dPrediction"
)

plt.grid(True)

plt.show()

## Summary

This example demonstrated the complete computational flow of a neural
network without training:

Forward:

input → dense layer → activation → prediction → loss

Backward:

loss derivative → activation gradient → dense parameter gradients

The model can now measure both:
- what prediction it produces,
- how each parameter contributed to the error.

The next step is optimization: using these gradients to update
parameters and reduce the loss over repeated iterations.